## 1. Install required packages

In [18]:
import cv2
import pyodbc
import numpy as np
from PIL import Image, ImageTk
from ultralytics import YOLO

print("Everything is installed ✅")

Everything is installed ✅


## 2. Import libraries

In [19]:
from pathlib import Path
from collections import defaultdict, deque
from datetime import datetime
import os
import threading
import tkinter as tk
from tkinter import ttk, filedialog, messagebox

import cv2
import numpy as np
import pyodbc
from PIL import Image, ImageTk
from ultralytics import YOLO

print("OpenCV:", cv2.__version__)
print("pyodbc:", pyodbc.version)


OpenCV: 5.0.0
pyodbc: 5.3.0


## 3. Project configuration

In [20]:
PROJECT_DIR = Path(
    r"D:\AI instant Training\cheating tracking"
)

VIDEO_DIR = PROJECT_DIR / "videos"
APP_OUTPUT_DIR = PROJECT_DIR / "desktop_app_outputs"
SCREENSHOT_DIR = APP_OUTPUT_DIR / "screenshots"

APP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)

DB_DRIVER = "ODBC Driver 18 for SQL Server"
DB_SERVER = r"(localdb)\MSSQLLocalDB"
DB_NAME = "ExamProctoringDB"

print("VIDEO_DIR:", VIDEO_DIR)
print("SCREENSHOT_DIR:", SCREENSHOT_DIR)
print("SQL Server:", DB_SERVER)
print("Database:", DB_NAME)


VIDEO_DIR: D:\AI instant Training\cheating tracking\videos
SCREENSHOT_DIR: D:\AI instant Training\cheating tracking\desktop_app_outputs\screenshots
SQL Server: (localdb)\MSSQLLocalDB
Database: ExamProctoringDB


## 4. AI thresholds

In [21]:
PERSON_CLASS = 0
CELL_PHONE_CLASS = 67

PERSON_CONF = 0.30
PHONE_CONF = 0.45
POSE_CONF = 0.30
POSE_STRIDE = 3

SIDE_LOOK_THRESHOLD_SEC = 2.5
HEAD_TURN_THRESHOLD = 3
HEAD_TURN_WINDOW_SEC = 8.0

PHONE_WINDOW_FRAMES = 15
PHONE_MIN_DETECTIONS = 8


## 5. ByteTrack configuration file

In [22]:
TRACKER_CONFIG = PROJECT_DIR / "bytetrack_exam.yaml"

tracker_yaml = '''tracker_type: bytetrack
track_high_thresh: 0.35
track_low_thresh: 0.10
new_track_thresh: 0.45
track_buffer: 60
match_thresh: 0.80
fuse_score: true
'''

TRACKER_CONFIG.write_text(
    tracker_yaml,
    encoding="utf-8"
)

print("Saved tracker config:", TRACKER_CONFIG)


Saved tracker config: D:\AI instant Training\cheating tracking\bytetrack_exam.yaml


## 6. SQL Server connection helper

In [23]:
def connection_string():
    return (
        f"DRIVER={{{DB_DRIVER}}};"
        f"SERVER={DB_SERVER};"
        f"DATABASE={DB_NAME};"
        "Trusted_Connection=yes;"
        "Encrypt=yes;"
        "TrustServerCertificate=yes;"
    )


def test_database_connection():
    conn = pyodbc.connect(
        connection_string()
    )

    cur = conn.cursor()

    cur.execute(
        "SELECT @@SERVERNAME, DB_NAME()"
    )

    row = cur.fetchone()

    conn.close()

    return {
        "server": row[0],
        "database": row[1]
    }


test_database_connection()


{'server': 'DESKTOP-M70NM43\\LOCALDB#6A7DD50D', 'database': 'ExamProctoringDB'}

## 7. Database helper class

In [24]:
class DatabaseManager:

    def __init__(self):
        self.conn = pyodbc.connect(
            connection_string()
        )

    def close(self):
        self.conn.close()

    def get_or_create_video(
        self,
        video_path,
        duration,
        fps,
        total_frames
    ):
        video_path = Path(
            video_path
        ).resolve()

        cur = self.conn.cursor()

        cur.execute(
            '''
            SELECT VideoID
            FROM dbo.ExamVideos
            WHERE VideoPath = ?
            ''',
            str(video_path)
        )

        row = cur.fetchone()

        if row:
            return int(row[0])

        cur.execute(
            '''
            INSERT INTO dbo.ExamVideos
            (
                VideoName,
                VideoPath,
                DurationSeconds,
                FPS,
                TotalFrames,
                Status
            )
            OUTPUT INSERTED.VideoID
            VALUES (?, ?, ?, ?, ?, ?)
            ''',
            video_path.name,
            str(video_path),
            float(duration),
            float(fps),
            int(total_frames),
            "Uploaded"
        )

        video_id = int(
            cur.fetchone()[0]
        )

        self.conn.commit()

        return video_id

    def update_video_status(
        self,
        video_id,
        status
    ):
        cur = self.conn.cursor()

        cur.execute(
            '''
            UPDATE dbo.ExamVideos
            SET Status = ?
            WHERE VideoID = ?
            ''',
            status,
            video_id
        )

        self.conn.commit()

    def insert_alert(
        self,
        video_id,
        event
    ):
        cur = self.conn.cursor()

        cur.execute(
            '''
            INSERT INTO dbo.CheatingEvents
            (
                VideoID,
                StudentTrackID,
                BehaviorDetected,
                ConfidenceScore,
                VideoTimeSeconds,
                EventDateTime,
                ScreenshotPath,
                HeadDirection,
                SideLookDuration,
                HeadTurnCount,
                PhoneHitsInWindow
            )
            OUTPUT INSERTED.EventID
            VALUES (?, ?, ?, ?, ?, SYSDATETIME(), ?, ?, ?, ?, ?)
            ''',
            int(video_id),
            int(event["student_id"]),
            str(event["behavior"]),
            float(event["confidence"]),
            float(event["video_time"]),
            str(event["screenshot_path"]),
            str(event["head_direction"]),
            float(event["side_look_duration"]),
            int(event["head_turn_count"]),
            int(event["phone_hits"])
        )

        event_id = int(
            cur.fetchone()[0]
        )

        self.conn.commit()

        return event_id

    def recent_alerts(
        self,
        limit=200
    ):
        cur = self.conn.cursor()

        cur.execute(
            f'''
            SELECT TOP {int(limit)}
                e.EventID,
                v.VideoName,
                e.StudentTrackID,
                e.BehaviorDetected,
                e.ConfidenceScore,
                e.VideoTimeSeconds,
                e.EventDateTime,
                e.ScreenshotPath
            FROM dbo.CheatingEvents e
            INNER JOIN dbo.ExamVideos v
                ON e.VideoID = v.VideoID
            ORDER BY e.EventID DESC
            '''
        )

        return cur.fetchall()


## 8. Geometry helpers

In [25]:
def box_center(box):
    x1, y1, x2, y2 = box
    return (
        (x1 + x2) / 2,
        (y1 + y2) / 2
    )


def expand_box(
    box,
    frame_shape,
    scale_x=0.25,
    scale_y=0.15
):
    h, w = frame_shape[:2]

    x1, y1, x2, y2 = map(
        float,
        box
    )

    bw = x2 - x1
    bh = y2 - y1

    return (
        max(0, x1 - bw * scale_x),
        max(0, y1 - bh * scale_y),
        min(w - 1, x2 + bw * scale_x),
        min(h - 1, y2 + bh * scale_y)
    )


def point_inside_box(
    point,
    box
):
    x, y = point
    x1, y1, x2, y2 = box

    return (
        x1 <= x <= x2
        and
        y1 <= y <= y2
    )


def box_iou(
    a,
    b
):
    ax1, ay1, ax2, ay2 = map(
        float,
        a
    )

    bx1, by1, bx2, by2 = map(
        float,
        b
    )

    x1 = max(ax1, bx1)
    y1 = max(ay1, by1)

    x2 = min(ax2, bx2)
    y2 = min(ay2, by2)

    iw = max(0, x2 - x1)
    ih = max(0, y2 - y1)

    intersection = iw * ih

    area_a = (
        max(0, ax2 - ax1)
        *
        max(0, ay2 - ay1)
    )

    area_b = (
        max(0, bx2 - bx1)
        *
        max(0, by2 - by1)
    )

    union = (
        area_a
        +
        area_b
        -
        intersection
    )

    return (
        intersection / union
        if union > 0
        else 0
    )


## 9. Head direction estimation

In [26]:
NOSE = 0
LEFT_EYE = 1
RIGHT_EYE = 2
LEFT_EAR = 3
RIGHT_EAR = 4


def estimate_head_direction(
    keypoints_xy,
    keypoints_conf=None
):
    xy = np.asarray(
        keypoints_xy,
        dtype=float
    )

    if (
        xy.ndim != 2
        or xy.shape[0] < 5
    ):
        return "unknown"

    if keypoints_conf is None:
        conf = np.ones(
            xy.shape[0],
            dtype=float
        )
    else:
        conf = np.asarray(
            keypoints_conf,
            dtype=float
        )

    def visible(
        index,
        threshold=0.25
    ):
        return (
            index < len(conf)
            and conf[index] >= threshold
            and xy[index, 0] > 0
            and xy[index, 1] > 0
        )

    if (
        visible(NOSE)
        and visible(LEFT_EYE)
        and visible(RIGHT_EYE)
    ):
        nose_x = xy[NOSE, 0]

        left_x = xy[
            LEFT_EYE,
            0
        ]

        right_x = xy[
            RIGHT_EYE,
            0
        ]

        midpoint = (
            left_x
            +
            right_x
        ) / 2

        eye_span = abs(
            left_x
            -
            right_x
        )

        if eye_span >= 3:

            offset = (
                nose_x
                -
                midpoint
            ) / eye_span

            if offset > 0.28:
                return "right"

            if offset < -0.28:
                return "left"

            return "forward"

    left_visible = (
        visible(LEFT_EYE)
        or visible(LEFT_EAR)
    )

    right_visible = (
        visible(RIGHT_EYE)
        or visible(RIGHT_EAR)
    )

    if (
        left_visible
        and not right_visible
    ):
        return "right"

    if (
        right_visible
        and not left_visible
    ):
        return "left"

    return "unknown"


## 10. Match pose to tracked students

In [27]:
def match_pose_to_tracks(
    students,
    pose_boxes,
    keypoints_xy,
    keypoints_conf
):
    directions = {}

    for pose_index, pose_box in enumerate(
        pose_boxes
    ):
        best_track_id = None
        best_iou = 0

        for student in students:

            score = box_iou(
                student["box"],
                pose_box
            )

            if score > best_iou:
                best_iou = score
                best_track_id = student[
                    "track_id"
                ]

        if (
            best_track_id is not None
            and best_iou >= 0.20
        ):
            conf = None

            if (
                keypoints_conf is not None
                and pose_index
                < len(keypoints_conf)
            ):
                conf = keypoints_conf[
                    pose_index
                ]

            directions[
                best_track_id
            ] = estimate_head_direction(
                keypoints_xy[
                    pose_index
                ],
                conf
            )

    return directions


## 11. Phone association

In [28]:
def phone_near_student(
    student_box,
    phones,
    frame_shape
):
    search_box = expand_box(
        student_box,
        frame_shape
    )

    best_phone = None
    best_conf = 0

    for phone in phones:

        if phone[
            "confidence"
        ] < PHONE_CONF:
            continue

        center = box_center(
            phone["box"]
        )

        if point_inside_box(
            center,
            search_box
        ):
            if (
                phone["confidence"]
                >
                best_conf
            ):
                best_conf = phone[
                    "confidence"
                ]
                best_phone = phone

    return best_phone


## 12. Student state

In [29]:
class StudentState:

    def __init__(self):

        self.last_direction = "unknown"

        self.side_started_at = None

        self.direction_changes = deque()

        self.phone_window = deque(
            maxlen=PHONE_WINDOW_FRAMES
        )

        self.phone_conf_window = deque(
            maxlen=PHONE_WINDOW_FRAMES
        )

        self.active_behavior = None


## 13. AI processor class

In [30]:
class AIProcessor:

    def __init__(self):

        print(
            "Loading YOLO11..."
        )

        self.tracker_model = YOLO(
            "yolo11n.pt"
        )

        print(
            "Loading YOLO11 Pose..."
        )

        self.pose_model = YOLO(
            "yolo11n-pose.pt"
        )

        self.states = defaultdict(
            StudentState
        )

        self.last_head_direction = {}

    def update_state(
        self,
        track_id,
        direction,
        phone_detected,
        phone_confidence,
        video_time
    ):
        state = self.states[
            track_id
        ]

        if direction in {
            "left",
            "right"
        }:

            if (
                state.last_direction
                in {"left", "right"}
                and
                direction
                != state.last_direction
            ):
                state.direction_changes.append(
                    video_time
                )

            while (
                state.direction_changes
                and
                video_time
                -
                state.direction_changes[0]
                >
                HEAD_TURN_WINDOW_SEC
            ):
                state.direction_changes.popleft()

            if (
                state.side_started_at
                is None
            ):
                state.side_started_at = (
                    video_time
                )

            side_duration = (
                video_time
                -
                state.side_started_at
            )

        else:

            state.side_started_at = None

            side_duration = 0

        if direction != "unknown":

            state.last_direction = (
                direction
            )

        state.phone_window.append(
            1
            if phone_detected
            else 0
        )

        state.phone_conf_window.append(
            float(
                phone_confidence
            )
            if phone_detected
            else 0
        )

        phone_hits = int(
            sum(
                state.phone_window
            )
        )

        confirmed_phone = (
            len(
                state.phone_window
            )
            ==
            PHONE_WINDOW_FRAMES
            and
            phone_hits
            >=
            PHONE_MIN_DETECTIONS
        )

        phone_conf = (
            max(
                state.phone_conf_window
            )
            if
            len(
                state.phone_conf_window
            )
            > 0
            else 0
        )

        return {
            "side_look_duration":
                side_duration,

            "head_turn_count":
                len(
                    state.direction_changes
                ),

            "phone_hits":
                phone_hits,

            "confirmed_phone":
                confirmed_phone,

            "phone_confidence":
                phone_conf
        }

    def make_decision(
        self,
        temporal
    ):

        if temporal[
            "confirmed_phone"
        ]:
            return {
                "is_cheating": True,
                "behavior":
                    "Phone detected",
                "confidence":
                    max(
                        0.80,
                        temporal[
                            "phone_confidence"
                        ]
                    )
            }

        if (
            temporal[
                "side_look_duration"
            ]
            >=
            SIDE_LOOK_THRESHOLD_SEC
        ):

            score = min(
                0.95,
                0.70
                +
                0.05
                *
                (
                    temporal[
                        "side_look_duration"
                    ]
                    -
                    SIDE_LOOK_THRESHOLD_SEC
                )
            )

            return {
                "is_cheating": True,
                "behavior":
                    "Prolonged sideways looking",
                "confidence":
                    score
            }

        if (
            temporal[
                "head_turn_count"
            ]
            >=
            HEAD_TURN_THRESHOLD
        ):

            return {
                "is_cheating": True,
                "behavior":
                    "Repeated head turns",
                "confidence":
                    0.75
            }

        return {
            "is_cheating": False,
            "behavior":
                "Normal",
            "confidence":
                0
        }

    def should_create_alert(
        self,
        track_id,
        decision
    ):

        state = self.states[
            track_id
        ]

        if not decision[
            "is_cheating"
        ]:

            state.active_behavior = None

            return False

        behavior = decision[
            "behavior"
        ]

        if (
            state.active_behavior
            is None
        ):

            state.active_behavior = (
                behavior
            )

            return True

        if (
            state.active_behavior
            !=
            behavior
        ):

            state.active_behavior = (
                behavior
            )

            return True

        return False


## 14. Frame processing logic

In [31]:
def process_ai_frame(
    processor,
    frame,
    frame_index,
    fps
):
    video_time = (
        frame_index / fps
        if fps > 0
        else 0
    )

    result = processor.tracker_model.track(
        source=frame,
        persist=True,
        tracker=str(
            TRACKER_CONFIG
        ),
        classes=[
            PERSON_CLASS,
            CELL_PHONE_CLASS
        ],
        conf=PERSON_CONF,
        verbose=False
    )[0]

    students = []
    phones = []

    if (
        result.boxes
        is not None
        and
        len(
            result.boxes
        ) > 0
    ):

        boxes = (
            result.boxes.xyxy
            .cpu()
            .numpy()
        )

        classes = (
            result.boxes.cls
            .int()
            .cpu()
            .tolist()
        )

        confidences = (
            result.boxes.conf
            .cpu()
            .tolist()
        )

        if (
            result.boxes.id
            is not None
        ):
            ids = (
                result.boxes.id
                .int()
                .cpu()
                .tolist()
            )
        else:
            ids = (
                [-1]
                *
                len(boxes)
            )

        for (
            box,
            class_id,
            confidence,
            track_id
        ) in zip(
            boxes,
            classes,
            confidences,
            ids
        ):

            item = {
                "box":
                    tuple(
                        map(
                            float,
                            box
                        )
                    ),

                "confidence":
                    float(
                        confidence
                    ),

                "track_id":
                    int(
                        track_id
                    )
            }

            if (
                class_id
                ==
                PERSON_CLASS
                and
                track_id >= 0
            ):
                students.append(
                    item
                )

            elif (
                class_id
                ==
                CELL_PHONE_CLASS
            ):
                phones.append(
                    item
                )

    if (
        frame_index
        %
        max(
            1,
            POSE_STRIDE
        )
        ==
        0
        and
        students
    ):

        pose_result = (
            processor.pose_model.predict(
                source=frame,
                conf=POSE_CONF,
                verbose=False
            )[0]
        )

        if (
            pose_result.boxes
            is not None
            and
            pose_result.keypoints
            is not None
        ):

            pose_boxes = (
                pose_result.boxes.xyxy
                .cpu()
                .numpy()
            )

            keypoints_xy = (
                pose_result.keypoints.xy
                .cpu()
                .numpy()
            )

            keypoints_conf = None

            if (
                pose_result.keypoints.conf
                is not None
            ):
                keypoints_conf = (
                    pose_result.keypoints.conf
                    .cpu()
                    .numpy()
                )

            directions = (
                match_pose_to_tracks(
                    students,
                    pose_boxes,
                    keypoints_xy,
                    keypoints_conf
                )
            )

            processor.last_head_direction.update(
                directions
            )

    annotated = frame.copy()

    events = []

    for phone in phones:

        if (
            phone["confidence"]
            <
            PHONE_CONF
        ):
            continue

        x1, y1, x2, y2 = map(
            int,
            phone["box"]
        )

        cv2.rectangle(
            annotated,
            (x1, y1),
            (x2, y2),
            (0, 165, 255),
            2
        )

        cv2.putText(
            annotated,
            f"Phone {phone['confidence']:.2f}",
            (
                x1,
                max(
                    20,
                    y1 - 8
                )
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 165, 255),
            2
        )

    for student in students:

        track_id = student[
            "track_id"
        ]

        direction = (
            processor
            .last_head_direction
            .get(
                track_id,
                "unknown"
            )
        )

        phone = phone_near_student(
            student["box"],
            phones,
            frame.shape
        )

        phone_detected = (
            phone is not None
        )

        phone_confidence = (
            float(
                phone[
                    "confidence"
                ]
            )
            if
            phone_detected
            else 0
        )

        temporal = (
            processor.update_state(
                track_id,
                direction,
                phone_detected,
                phone_confidence,
                video_time
            )
        )

        decision = (
            processor.make_decision(
                temporal
            )
        )

        x1, y1, x2, y2 = map(
            int,
            student["box"]
        )

        if decision[
            "is_cheating"
        ]:

            color = (
                0,
                0,
                255
            )

            status = (
                "SUSPECTED CHEATING"
            )

        else:

            color = (
                0,
                255,
                0
            )

            status = (
                "NORMAL"
            )

        cv2.rectangle(
            annotated,
            (x1, y1),
            (x2, y2),
            color,
            3
        )

        cv2.putText(
            annotated,
            f"Student {track_id} | {status}",
            (
                x1,
                max(
                    20,
                    y1 - 28
                )
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.58,
            color,
            2
        )

        cv2.putText(
            annotated,
            (
                f"{decision['behavior']} "
                f"| head:{direction}"
            ),
            (
                x1,
                max(
                    20,
                    y1 - 7
                )
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.48,
            color,
            2
        )

        if (
            processor
            .should_create_alert(
                track_id,
                decision
            )
        ):

            events.append({
                "student_id":
                    track_id,

                "behavior":
                    decision[
                        "behavior"
                    ],

                "confidence":
                    decision[
                        "confidence"
                    ],

                "video_time":
                    video_time,

                "head_direction":
                    direction,

                "side_look_duration":
                    temporal[
                        "side_look_duration"
                    ],

                "head_turn_count":
                    temporal[
                        "head_turn_count"
                    ],

                "phone_hits":
                    temporal[
                        "phone_hits"
                    ]
            })

    return (
        annotated,
        events
    )


## 15. Screenshot helper

In [32]:
def save_screenshot(
    frame,
    video_name,
    event
):
    safe_behavior = (
        event[
            "behavior"
        ]
        .lower()
        .replace(
            " ",
            "_"
        )
    )

    timestamp = (
        datetime.now()
        .strftime(
            "%Y%m%d_%H%M%S_%f"
        )
    )

    filename = (
        f"{Path(video_name).stem}"
        f"_student_{event['student_id']}"
        f"_{safe_behavior}"
        f"_{timestamp}.jpg"
    )

    screenshot_path = (
        SCREENSHOT_DIR
        /
        filename
    )

    cv2.imwrite(
        str(
            screenshot_path
        ),
        frame
    )

    return screenshot_path


## 16. Tkinter Desktop App

This cell defines the final application.


In [33]:
class ExamProctoringApp:

    def __init__(self):

        self.root = tk.Tk()

        self.root.title(
            "AI Exam Proctoring System"
        )

        self.root.geometry(
            "1280x820"
        )

        self.selected_video = None

        self.video_id = None

        self.worker = None

        self.stop_event = (
            threading.Event()
        )

        self.processor = None

        self.current_photo = None

        self.status_var = (
            tk.StringVar(
                value="Ready"
            )
        )

        self.video_var = (
            tk.StringVar(
                value=
                    "No video selected"
            )
        )

        self.progress_var = (
            tk.DoubleVar(
                value=0
            )
        )

        self.build_ui()

        self.load_alert_history()

    def build_ui(self):

        top = ttk.Frame(
            self.root,
            padding=10
        )

        top.pack(
            fill="x"
        )

        ttk.Button(
            top,
            text="Upload Video",
            command=self.upload_video
        ).pack(
            side="left",
            padx=(0, 8)
        )

        self.start_button = (
            ttk.Button(
                top,
                text="Start Processing",
                command=
                    self.start_processing
            )
        )

        self.start_button.pack(
            side="left",
            padx=(0, 8)
        )

        self.stop_button = (
            ttk.Button(
                top,
                text="Stop",
                command=
                    self.stop_processing,
                state="disabled"
            )
        )

        self.stop_button.pack(
            side="left",
            padx=(0, 8)
        )

        ttk.Button(
            top,
            text="Test DB",
            command=
                self.test_database
        ).pack(
            side="left",
            padx=(0, 8)
        )

        ttk.Button(
            top,
            text="Refresh Alerts",
            command=
                self.load_alert_history
        ).pack(
            side="left"
        )

        ttk.Label(
            top,
            textvariable=
                self.video_var
        ).pack(
            side="left",
            padx=15
        )

        ttk.Label(
            top,
            textvariable=
                self.status_var
        ).pack(
            side="right"
        )

        ttk.Progressbar(
            self.root,
            variable=
                self.progress_var,
            maximum=100
        ).pack(
            fill="x",
            padx=10,
            pady=(0, 8)
        )

        pane = ttk.Panedwindow(
            self.root,
            orient=tk.VERTICAL
        )

        pane.pack(
            fill="both",
            expand=True,
            padx=10,
            pady=(0, 10)
        )

        video_frame = (
            ttk.LabelFrame(
                pane,
                text=
                    "Live Processing",
                padding=8
            )
        )

        alerts_frame = (
            ttk.LabelFrame(
                pane,
                text=
                    "Alert History",
                padding=8
            )
        )

        pane.add(
            video_frame,
            weight=3
        )

        pane.add(
            alerts_frame,
            weight=2
        )

        self.video_label = (
            ttk.Label(
                video_frame,
                text=
                    "Upload a video to start",
                anchor="center"
            )
        )

        self.video_label.pack(
            fill="both",
            expand=True
        )

        columns = (
            "event",
            "video",
            "student",
            "behavior",
            "confidence",
            "video_time",
            "datetime",
            "screenshot"
        )

        self.tree = (
            ttk.Treeview(
                alerts_frame,
                columns=columns,
                show="headings"
            )
        )

        headings = {
            "event":
                "Event ID",

            "video":
                "Video",

            "student":
                "Student ID",

            "behavior":
                "Behavior",

            "confidence":
                "Confidence",

            "video_time":
                "Video Time",

            "datetime":
                "Date / Time",

            "screenshot":
                "Screenshot"
        }

        for column in columns:

            self.tree.heading(
                column,
                text=
                    headings[
                        column
                    ]
            )

        self.tree.pack(
            fill="both",
            expand=True
        )

        self.tree.bind(
            "<Double-1>",
            self.open_screenshot
        )

    def test_database(self):

        try:

            result = (
                test_database_connection()
            )

            messagebox.showinfo(
                "SQL Server",
                (
                    "Connected successfully\n\n"
                    f"Server: {result['server']}\n"
                    f"Database: {result['database']}"
                )
            )

        except Exception as exc:

            messagebox.showerror(
                "Database Error",
                str(exc)
            )

    def upload_video(self):

        path = (
            filedialog
            .askopenfilename(
                initialdir=
                    str(
                        VIDEO_DIR
                    ),
                filetypes=[
                    (
                        "Video Files",
                        "*.mp4 *.avi *.mov *.mkv"
                    )
                ]
            )
        )

        if not path:
            return

        video_path = Path(
            path
        )

        cap = cv2.VideoCapture(
            str(
                video_path
            )
        )

        fps = cap.get(
            cv2.CAP_PROP_FPS
        )

        total_frames = int(
            cap.get(
                cv2.CAP_PROP_FRAME_COUNT
            )
        )

        duration = (
            total_frames / fps
            if fps > 0
            else 0
        )

        ok, frame = cap.read()

        cap.release()

        db = DatabaseManager()

        self.video_id = (
            db.get_or_create_video(
                video_path,
                duration,
                fps,
                total_frames
            )
        )

        db.close()

        self.selected_video = (
            video_path
        )

        self.video_var.set(
            (
                f"{video_path.name}"
                f" | VideoID:"
                f" {self.video_id}"
            )
        )

        if ok:

            self.update_frame(
                frame
            )

    def start_processing(self):

        if (
            self.selected_video
            is None
        ):

            messagebox.showwarning(
                "Video",
                "Upload a video first."
            )

            return

        if (
            self.worker
            and
            self.worker.is_alive()
        ):
            return

        self.stop_event.clear()

        self.start_button.configure(
            state="disabled"
        )

        self.stop_button.configure(
            state="normal"
        )

        self.worker = (
            threading.Thread(
                target=
                    self.process_video,
                daemon=True
            )
        )

        self.worker.start()

    def stop_processing(self):

        self.stop_event.set()

        self.status_var.set(
            "Stopping..."
        )

    def process_video(self):

        cap = None
        db = None

        try:

            self.root.after(
                0,
                self.status_var.set,
                "Loading AI models..."
            )

            self.processor = (
                AIProcessor()
            )

            db = DatabaseManager()

            db.update_video_status(
                self.video_id,
                "Processing"
            )

            cap = cv2.VideoCapture(
                str(
                    self.selected_video
                )
            )

            fps = cap.get(
                cv2.CAP_PROP_FPS
            )

            total_frames = int(
                cap.get(
                    cv2.CAP_PROP_FRAME_COUNT
                )
            )

            frame_index = 0

            while (
                not
                self.stop_event.is_set()
            ):

                ok, frame = cap.read()

                if not ok:
                    break

                (
                    annotated,
                    events
                ) = process_ai_frame(
                    self.processor,
                    frame,
                    frame_index,
                    fps
                )

                for event in events:

                    screenshot_path = (
                        save_screenshot(
                            annotated,
                            self.selected_video.name,
                            event
                        )
                    )

                    event[
                        "screenshot_path"
                    ] = str(
                        screenshot_path
                    )

                    event_id = (
                        db.insert_alert(
                            self.video_id,
                            event
                        )
                    )

                    event[
                        "event_id"
                    ] = event_id

                    self.root.after(
                        0,
                        self.handle_alert,
                        event
                    )

                self.root.after(
                    0,
                    self.update_frame,
                    annotated.copy()
                )

                if (
                    total_frames
                    > 0
                ):

                    progress = (
                        100
                        *
                        (
                            frame_index + 1
                        )
                        /
                        total_frames
                    )

                    self.root.after(
                        0,
                        self.progress_var.set,
                        progress
                    )

                frame_index += 1

            status = (
                "Stopped"
                if
                self.stop_event.is_set()
                else
                "Processed"
            )

            db.update_video_status(
                self.video_id,
                status
            )

            self.root.after(
                0,
                self.status_var.set,
                status
            )

        except Exception as exc:

            self.root.after(
                0,
                lambda:
                    messagebox.showerror(
                        "Processing Error",
                        str(exc)
                    )
            )

        finally:

            if cap is not None:
                cap.release()

            if db is not None:
                db.close()

            self.root.after(
                0,
                lambda:
                    self.start_button.configure(
                        state="normal"
                    )
            )

            self.root.after(
                0,
                lambda:
                    self.stop_button.configure(
                        state="disabled"
                    )
            )

    def handle_alert(
        self,
        event
    ):

        self.root.bell()

        self.status_var.set(
            (
                "ALERT: "
                f"Student {event['student_id']} "
                f"- {event['behavior']}"
            )
        )

        self.load_alert_history()

    def load_alert_history(self):

        try:

            db = DatabaseManager()

            rows = db.recent_alerts(
                200
            )

            db.close()

            for item in (
                self.tree.get_children()
            ):
                self.tree.delete(
                    item
                )

            for row in rows:

                self.tree.insert(
                    "",
                    "end",
                    values=(
                        row.EventID,
                        row.VideoName,
                        row.StudentTrackID,
                        row.BehaviorDetected,
                        round(
                            float(
                                row.ConfidenceScore
                                or 0
                            ),
                            2
                        ),
                        round(
                            float(
                                row.VideoTimeSeconds
                                or 0
                            ),
                            2
                        ),
                        row.EventDateTime,
                        row.ScreenshotPath
                    )
                )

        except Exception:
            pass

    def update_frame(
        self,
        frame
    ):

        h, w = frame.shape[
            :2
        ]

        max_width = 900

        if w > max_width:

            scale = (
                max_width / w
            )

            frame = cv2.resize(
                frame,
                (
                    int(
                        w
                        *
                        scale
                    ),
                    int(
                        h
                        *
                        scale
                    )
                )
            )

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        image = Image.fromarray(
            rgb
        )

        self.current_photo = (
            ImageTk.PhotoImage(
                image=image
            )
        )

        self.video_label.configure(
            image=
                self.current_photo,
            text=""
        )

    def open_screenshot(
        self,
        event=None
    ):

        selected = (
            self.tree.selection()
        )

        if not selected:
            return

        values = (
            self.tree.item(
                selected[0],
                "values"
            )
        )

        screenshot_path = (
            values[7]
        )

        if (
            screenshot_path
            and
            os.path.exists(
                screenshot_path
            )
        ):

            os.startfile(
                screenshot_path
            )

        else:

            messagebox.showwarning(
                "Screenshot",
                "Screenshot file was not found."
            )

    def run(self):

        self.root.mainloop()


## 17. Launch the Desktop Application

Run the next cell only when you are ready to open the Tkinter window.

> In VS Code/Jupyter, the cell will remain active while the desktop window is open.


In [ ]:
app = ExamProctoringApp()
app.run()


Loading YOLO11...
Loading YOLO11 Pose...


# Final workflow

```text
Tkinter
   ↓
Upload Video
   ↓
YOLO11 + ByteTrack
   ↓
Pose + Phone Detection
   ↓
Behavior Decision
   ↓
Normal → Green
Cheating → Red
   ↓
Screenshot
   ↓
SQL Server
   ↓
Alert History
```

## Local screenshot folder

```text
D:\AI instant Training\cheating tracking\desktop_app_outputs\screenshots
```

## Database

```text
Server:
(localdb)\MSSQLLocalDB

Database:
ExamProctoringDB
```

## Important note

Older alerts imported from Colab may still contain paths beginning with:

```text
/content/drive/...
```

Those old screenshot paths cannot be opened directly on Windows.

New alerts created through this Tkinter app will contain valid Windows screenshot paths.
